# 04 — Citation Alignment (the cosine matrix)

**Audience:** Engineers / SEs — this is the "open the black box" notebook.

**What this notebook shows:**

1. **`cite_align` in isolation.** Notebook 03 ended by calling
   `cite_answer(gen_B, res_dual)` and pulling a `CitedAnswer` out
   the other side. Here we do that by hand, step by step: split the
   answer into sentences, embed both sides, build the cosine
   similarity matrix, apply `tau` + `top_k`. Nothing magic.
2. **τ sensitivity.** Sweep `tau ∈ {0.55, 0.65, 0.75, 0.85}` against
   a fixed retrieval pool + fixed answer. Shows the precision /
   recall knob the aligner gives you.
3. **`top_k` sensitivity.** Same idea — how many candidates per
   answer sentence you're willing to credit.
4. **The cosine heatmap.** Answer-sentences × candidate-sentences.
   This is the picture that justifies the whole approach — and
   also shows you exactly where it will fail.
5. **A failure example.** A case where cosine picks a *parallel*
   sentence (same topic, different source) over the author's intended
   sid. This is the payoff — it's why faithfulness + self-consistency
   matter downstream (notebook 05 / 06).

**Prerequisites:**
- Notebook 03 completed **OR** `RESUME_FROM_CACHE=True` (we reuse
  `data/notebook_cache/retrieval/snapshot.json` and a hand-written
  clean answer, because the aligner math is deterministic given a
  fixed pool + answer).
- `.env` with Azure OpenAI embeddings creds — unless you're running
  the synthetic failure example only, in which case nothing Azure is
  needed (we pass pre-computed embeddings).


In [ ]:
# --- Top-level knobs --------------------------------------------------------
# 04 defaults to cache because the aligner is a pure function of
# (retrieval pool, answer_text). No need to re-pay the RAG call.
RESUME_FROM_CACHE = True

# τ sweep for Part 2. Spans the practical range: 0.55 is "very
# permissive", 0.85 is "almost exact paraphrase".
TAU_VALUES = [0.55, 0.65, 0.75, 0.85]

# top_k sweep for Part 3.
TOP_K_VALUES = [1, 3, 5]

# Hand-written clean answer to align against. Written from the snapshot
# query ("What type of payment must be reported as income if received by
# a dependent care provider?") so the aligner has something substantive
# to match. If the cache load fails, this doubles as the fallback text.
FALLBACK_ANSWER = (
    "If you provide childcare, whether in the child's home or in your own "
    "home, the pay you receive must be included in your income. "
    "Babysitters are covered by the same rules, even when paid only "
    "occasionally. If you are not an employee, the payments are treated "
    "as self-employment income and reported on Schedule C."
)

# Snapshot path — serialized RetrievalResult from notebook 03's dual run.
SNAPSHOT_PATH = "data/notebook_cache/retrieval/snapshot.json"


## Setup — imports, cfg, cache loader

Same import style as notebook 03. `RetrievalResult` / `SentenceHit` /
`ChunkHit` round-trip through plain dicts, so we deserialize the
snapshot by hand (the dataclasses don't ship a `from_dict`).

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

load_dotenv(REPO_ROOT / ".env")

from sentcite.config import AzureConfig
from sentcite.retrieval import retrieve, RetrievalResult, ChunkHit, SentenceHit
from sentcite.generate import generate, GenerateOutput
from sentcite.cite_align import (
    cite_answer,
    align_post_generation,
    _candidate_pool,  # private, but this notebook is about opening the box
)
from sentcite.chunking import split_sentences
from sentcite.indexing import embed_texts

# cfg is only needed if we fall back to live calls (embeddings or RAG).
try:
    cfg = AzureConfig.from_env()
except Exception as e:  # missing .env etc.
    cfg = None
    print(f"[warn] AzureConfig unavailable ({e!r}) — cache path only.")

RESUME_FROM_CACHE, bool(cfg)


In [ ]:
def _result_from_dict(d: dict) -> RetrievalResult:
    """Rehydrate a RetrievalResult from the notebook 03 snapshot dict."""
    chunks = [
        ChunkHit(
            chunk_id=c["chunk_id"],
            document_id=c["document_id"],
            page=int(c.get("page") or 0),
            section_path=list(c.get("section_path") or []),
            text=c.get("text") or "",
            token_count=int(c.get("token_count") or 0),
            sentences=list(c.get("sentences") or []),
            source=c.get("source") or "chunk_search",
            score=c.get("score"),
            reranker=c.get("reranker"),
        )
        for c in d.get("chunks", [])
    ]
    cands = [
        SentenceHit(
            sentence_id=s["sentence_id"],
            chunk_id=s["chunk_id"],
            document_id=s["document_id"],
            page=int(s.get("page") or 0),
            section_path=list(s.get("section_path") or []),
            text=s.get("text") or "",
            score=s.get("score"),
            reranker=s.get("reranker"),
        )
        for s in d.get("sentence_candidates", [])
    ]
    return RetrievalResult(
        query=d.get("query", ""),
        mode=d.get("mode", "dual"),
        chunks=chunks,
        sentence_candidates=cands,
        latency_ms=float(d.get("latency_ms") or 0.0),
        chunk_search_hits=int(d.get("chunk_search_hits") or 0),
        sentence_search_hits=int(d.get("sentence_search_hits") or 0),
        parent_chunks_added=int(d.get("parent_chunks_added") or 0),
    )


def _load_pool_and_answer():
    """Return (question, RetrievalResult, answer_text). Cache-first."""
    snap_path = REPO_ROOT / SNAPSHOT_PATH
    if RESUME_FROM_CACHE and snap_path.exists():
        d = json.loads(snap_path.read_text())
        question = d.get("query", "")
        result = _result_from_dict(d)
        # The snapshot doesn't carry the clean answer from Strategy B
        # (it's a retrieval dump). Use the hand-written fallback.
        return question, result, FALLBACK_ANSWER

    if cfg is None:
        raise RuntimeError(
            "Cache miss and no AzureConfig — set RESUME_FROM_CACHE=True "
            "and ensure data/notebook_cache/retrieval/snapshot.json exists."
        )
    # Live path.
    question = (
        "What type of payment must be reported as income if received "
        "by a dependent care provider?"
    )
    result = retrieve(question, cfg=cfg, mode="dual", k_chunks=5, k_sentences=20)
    gen = generate(question, result, strategy="post_gen_alignment", cfg=cfg)
    return question, result, gen.answer_text


QUESTION, RESULT, ANSWER_TEXT = _load_pool_and_answer()
pool = _candidate_pool(RESULT)
print(f"question : {QUESTION}")
print(f"pool     : {len(pool)} candidate sentences")
print(f"answer   : {ANSWER_TEXT}")


## Part 1 — What `cite_align` does, visually

Three steps, nothing fancy:

1. **Split** the answer into sentences with the same `split_sentences`
   helper `align_post_generation` uses (spaCy under the hood).
2. **Embed** both sides with `embed_texts` (Azure OpenAI
   `text-embedding-3-large`) — same helper the aligner calls
   internally, so the vectors we plot *are* the vectors the aligner
   would use.
3. **Cosine** matrix = L2-normalized answer matrix @ L2-normalized
   candidate matrix transposed.

The heatmap is the whole story: bright cells are candidate sentences
the aligner will consider; τ is a horizontal cutoff; `top_k` is
"keep the top k per row".

In [ ]:
# Step 1 — split the answer using the same splitter cite_align uses.
answer_sents = split_sentences(ANSWER_TEXT)  # list of (start, end, text)
answer_texts = [t for _, _, t in answer_sents]
print(f"{len(answer_texts)} answer sentences:")
for i, t in enumerate(answer_texts):
    print(f"  [{i}] {t}")


In [ ]:
# Step 2 — embed both sides. Uses live embeddings by default; if you
# want a fully-offline run you can stash these to a .npy once computed.
# The aligner L2-normalizes internally, so we do the same here.
def _l2(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    return mat / np.where(norms < 1e-12, 1.0, norms)


if cfg is None:
    raise RuntimeError(
        "Need AzureConfig for Part 1 embeddings. Provide .env creds, "
        "or skip to Part 4 (synthetic example) which supplies vectors."
    )

A_vecs = np.asarray(embed_texts(answer_texts, cfg=cfg), dtype=np.float32)
C_vecs = np.asarray(embed_texts([s.text for s in pool], cfg=cfg), dtype=np.float32)
A_norm = _l2(A_vecs)
C_norm = _l2(C_vecs)

# Step 3 — cosine similarity matrix: (n_answer, n_candidates).
SIMS = A_norm @ C_norm.T
print(f"cosine matrix shape: {SIMS.shape}  (answer_sents × candidates)")
print(f"min={SIMS.min():.3f}  max={SIMS.max():.3f}  mean={SIMS.mean():.3f}")


In [ ]:
# Render as a heatmap. Prefer matplotlib; gracefully degrade to a
# pandas-styled DataFrame if plotting isn't available in this env.
row_labels = [f"A{i}: {t[:50]}{'…' if len(t) > 50 else ''}" for i, t in enumerate(answer_texts)]
col_labels = [s.sentence_id for s in pool]

sim_df = pd.DataFrame(SIMS, index=row_labels, columns=col_labels)

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(max(8, 0.45 * len(col_labels)), max(3, 0.7 * len(row_labels))))
    im = ax.imshow(SIMS, aspect="auto", cmap="viridis", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=75, ha="right", fontsize=7)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_xlabel("candidate sentence_id")
    ax.set_ylabel("answer sentence")
    ax.set_title(f"cosine(answer, candidates) — τ=0.65 cutoff shown as contour")
    # Overlay the τ=0.65 mask as a thin highlight.
    for i in range(SIMS.shape[0]):
        for j in range(SIMS.shape[1]):
            if SIMS[i, j] >= 0.65:
                ax.text(j, i, f"{SIMS[i,j]:.2f}", ha="center", va="center",
                        fontsize=6, color="white")
    fig.colorbar(im, ax=ax, shrink=0.8, label="cosine")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"[matplotlib unavailable: {e!r}] — falling back to styled DataFrame")
    display(sim_df.style.background_gradient(cmap="viridis", axis=None).format("{:.2f}"))


In [ ]:
# Top-3 candidates per answer sentence at τ=0.65 — this is what the
# aligner would emit with the default knobs.
rows = []
for i, at in enumerate(answer_texts):
    order = np.argsort(-SIMS[i])
    kept = [(int(j), float(SIMS[i, j])) for j in order if SIMS[i, j] >= 0.65][:3]
    rows.append({
        "answer_sent": f"A{i}: {at[:70]}{'…' if len(at) > 70 else ''}",
        "top_cites": ", ".join(f"{pool[j].sentence_id}({s:.2f})" for j, s in kept) or "(none — below τ)",
    })
pd.DataFrame(rows)


## Part 2 — τ sweep

Same pool, same answer. Only `tau` changes. We call
`align_post_generation` directly (not `cite_answer`) so we can pass
the pre-computed embeddings and avoid re-paying embeddings per τ.

**Read the table as:** lower τ pulls more candidates in (more
`avg_cites_per_sentence`, fewer `uncited`), higher τ is more
conservative. 0.65 is the default in evaluation — see notebook 05
for the precision/recall curve that justifies it.

In [ ]:
def _make_gen_stub(answer_text: str) -> GenerateOutput:
    """Minimal GenerateOutput to feed align_post_generation."""
    return GenerateOutput(
        question=QUESTION,
        strategy="post_gen_alignment",
        answer_text=answer_text,
        model="cached",
        retrieved_chunk_ids=RESULT.candidate_chunk_ids(),
        context_sentence_ids=RESULT.candidate_sentence_ids(),
    )


GEN_STUB = _make_gen_stub(ANSWER_TEXT)


def _summarize(cited) -> dict:
    n_cited = sum(1 for s in cited.sentences if s.citations)
    n_uncited = sum(1 for s in cited.sentences if not s.citations)
    total_cites = sum(len(s.citations) for s in cited.sentences)
    return {
        "num_cited_sentences": n_cited,
        "num_uncited": n_uncited,
        "total_citations": total_cites,
        "avg_cites_per_sentence": round(total_cites / max(1, len(cited.sentences)), 2),
    }


tau_rows = []
for tau in TAU_VALUES:
    cited = align_post_generation(
        GEN_STUB, RESULT,
        cfg=cfg, tau=tau, top_k=3,
        answer_embeddings=A_vecs.tolist(),
        candidate_embeddings=C_vecs.tolist(),
    )
    tau_rows.append({"tau": tau, "top_k": 3, **_summarize(cited)})

pd.DataFrame(tau_rows)


## Part 3 — `top_k` sweep

`tau` fixed at 0.65. `top_k` is the per-answer-sentence cap on kept
citations. In practice `top_k=3` is the sweet spot: the model's
answer sentences usually have 1–2 real sources, and 3 gives the
margin for genuine multi-source claims without letting the long
tail in.

In [ ]:
topk_rows = []
for k in TOP_K_VALUES:
    cited = align_post_generation(
        GEN_STUB, RESULT,
        cfg=cfg, tau=0.65, top_k=k,
        answer_embeddings=A_vecs.tolist(),
        candidate_embeddings=C_vecs.tolist(),
    )
    topk_rows.append({"tau": 0.65, "top_k": k, **_summarize(cited)})

pd.DataFrame(topk_rows)


## Part 4 — A failure example (the payoff cell)

Cosine similarity is a *surface* signal. It doesn't know which
source the author actually read to write the answer sentence — only
which source is most lexically/semantically close. When two sources
in the pool say nearly the same thing (think: cross-references
between IRS pubs), the aligner can pick the wrong one.

Below we construct a tiny synthetic pool where:

- `correct-001` is the author's intended source (from Pub 17).
- `parallel-099` is a **parallel** sentence from a different
  publication that paraphrases the same rule (Pub 926 cross-ref).
- `distractor-042` is on-topic but not the rule being cited.

We feed all three into `align_post_generation` with pre-computed
embeddings we control, then inspect what the aligner picks. This is
the pattern the eval harness (notebook 05) catches via *faithfulness*
(does the cited sentence entail the answer sentence?) and via
*self-consistency* (do repeated generations agree on the cite?).

In [ ]:
from sentcite.schema import CitedAnswer  # noqa — only for type context

# Synthetic answer sentence and synthetic pool.
SYN_ANSWER = "Childcare providers must include the pay they receive in their income."

SYN_POOL = [
    SentenceHit(
        sentence_id="correct-001",
        chunk_id="pub17-c09",
        document_id="irs_pub_17",
        page=49,
        section_path=["Pub 17", "Employee Compensation", "Childcare providers"],
        text=(
            "Childcare providers. If you provide childcare, either in the "
            "child's home or in your home or other place of business, the "
            "pay you receive must be included in your income."
        ),
    ),
    SentenceHit(
        sentence_id="parallel-099",
        chunk_id="pub926-c02",
        document_id="irs_pub_926",
        page=7,
        section_path=["Pub 926", "Household Employer", "Paying a Provider"],
        text=(
            "A childcare provider is required to report the payments they "
            "receive for providing childcare as income on their tax return."
        ),
    ),
    SentenceHit(
        sentence_id="distractor-042",
        chunk_id="pub503-c05",
        document_id="irs_pub_503",
        page=12,
        section_path=["Pub 503", "Care Provider Information"],
        text=(
            "You must identify the care provider on your return, including "
            "their name, address, and taxpayer identification number."
        ),
    ),
]

# Build a synthetic RetrievalResult wrapping the pool only (no chunks
# needed — _candidate_pool prefers sentence_candidates when present).
SYN_RESULT = RetrievalResult(
    query="synthetic — parallel-source failure case",
    mode="dual",
    chunks=[],
    sentence_candidates=SYN_POOL,
)

# Embed once (live). If Azure is not available, we fall back to the
# answer-vs-pool similarities computed analytically from the real run
# above would not apply — so we require embeddings here.
if cfg is None:
    raise RuntimeError("Part 4 requires embeddings. Provide .env creds.")

SYN_A = np.asarray(embed_texts([SYN_ANSWER], cfg=cfg), dtype=np.float32)
SYN_C = np.asarray(embed_texts([s.text for s in SYN_POOL], cfg=cfg), dtype=np.float32)
SYN_A_n = _l2(SYN_A)
SYN_C_n = _l2(SYN_C)
syn_sims = (SYN_A_n @ SYN_C_n.T)[0]  # shape (3,)

pd.DataFrame([
    {
        "sentence_id": SYN_POOL[j].sentence_id,
        "doc": SYN_POOL[j].document_id,
        "cosine_vs_answer": round(float(syn_sims[j]), 4),
        "text": SYN_POOL[j].text,
    }
    for j in np.argsort(-syn_sims)
])


In [ ]:
# Run the real aligner and see who wins.
syn_gen = GenerateOutput(
    question="What do childcare providers do with the pay they receive?",
    strategy="post_gen_alignment",
    answer_text=SYN_ANSWER,
    model="synthetic",
    retrieved_chunk_ids=[],
    context_sentence_ids=[s.sentence_id for s in SYN_POOL],
)

syn_cited = align_post_generation(
    syn_gen, SYN_RESULT,
    cfg=cfg, tau=0.65, top_k=3,
    answer_embeddings=SYN_A.tolist(),
    candidate_embeddings=SYN_C.tolist(),
)

for s in syn_cited.sentences:
    print(f"answer: {s.text}")
    for c in s.citations:
        print(f"  → [{c.sentence_id}] conf={c.confidence:.3f}  doc={c.document_id}")
    if not s.citations:
        print("  (no citations above τ)")


**What happened.** If `parallel-099` scores ≥ `correct-001` (it
often does — it's a cleaner paraphrase of the *same rule*), the
aligner will list it first, even though the author was writing
from `correct-001`. Strict `sid == gold_sid` evaluation penalizes
this; *faithfulness* evaluation (does the cited sentence support
the claim?) forgives it, because both sources do support the claim.
This tension — sid-accuracy vs. faithfulness — is exactly what
notebook 05 quantifies, and why we keep both metrics.

The aligner is a *retrieval* step, not a *grounding* step. The
harness on top is what turns its candidates into trustworthy
citations.

## Part 5 — Takeaways

- **`cite_align` is good at:** finding *at least one* plausible
  source sentence for most well-formed answer claims, with no
  dependence on the RAG model's willingness to emit tags. It's a
  dependable floor.
- **`cite_align` is not good at:** distinguishing between
  near-duplicate sources ("the author meant *this* sid, not *that*
  one"). Cosine over sentence embeddings can't see provenance.
- **Knobs that matter:** `tau` trades recall for precision (low τ
  → more citations, more noise); `top_k` caps per-claim fan-out.
  Defaults (`τ=0.65`, `top_k=3`) balance both for the eval corpus.
- **How the harness compensates:** faithfulness (NLI-style entail
  check) forgives parallel-source swaps when the claim is still
  supported; self-consistency catches sentences that *no* run can
  confidently cite; hit@k@τ tracks sid-accuracy when you do care
  about exact provenance.


## What's next

- **`05_evaluation.ipynb`** — runs the full eval harness over
  synthetic GT: citation hit@k, faithfulness via an LLM judge,
  self-consistency across N generations, and the τ/top-k precision
  / recall curves that justify the defaults used here.
- **`05a_synthetic_gt.ipynb`** (already in repo) — how the GT
  pairs (question, gold_sid) were generated in the first place.
